# 02 — Exploratory analysis

Uses only the locally stored processed dataset generated by **01_cleaning**. This notebook describes score distributions, coding-task frequencies, and preliminary associations. **Formal hypothesis tests remain in Notebook 03.** No participant-level data are printed or exported here.

Run from a fresh kernel, top to bottom. The exploratory statistics must not be adjusted to produce a desired conclusion.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
processed_file = project_root / "data" / "processed" / "survey_processed.csv"
if not processed_file.is_file():
    raise FileNotFoundError(f"Missing {processed_file}. Run Notebook 01 first.")
df = pd.read_csv(processed_file)

core_variables = ["gad7_total", "aise_mean", "llm_overall", "llm_task_mean"]
llm_task_variables = ["llm_debug", "llm_errors", "llm_generate", "llm_refactor", "llm_validate"]
analysis_variables = core_variables + llm_task_variables
missing = sorted(set(analysis_variables) - set(df.columns))
if missing:
    raise ValueError(f"Required columns missing: {missing}")
for col in analysis_variables:
    df[col] = pd.to_numeric(df[col], errors="raise")
if df[analysis_variables].isna().any().any():
    raise ValueError("Core analysis variables have missing values; inspect Notebook 01 before continuing.")
if len(df) != 124:
    raise ValueError(f"Expected 124 complete survey records, found {len(df)}. Investigate data selection.")
for col, low, high in [("gad7_total", 0, 21), ("aise_mean", 1, 7), ("llm_overall", 1, 5), ("llm_task_mean", 1, 5)]:
    if not df[col].between(low, high).all():
        raise ValueError(f"Out-of-range values in {col}")
for col in llm_task_variables:
    if not df[col].isin(range(1, 6)).all():
        raise ValueError(f"Invalid response categories in {col}")
if not np.allclose(df[llm_task_variables].mean(axis=1), df["llm_task_mean"], atol=1e-10):
    raise ValueError("Task-frequency composite disagrees with mean of its five items")
print(f"Verified N={len(df)}; all nine analysis variables present, complete and within expected ranges.")

In [ ]:
# Aggregate descriptive statistics, retaining precision for downstream comparisons.
def summary_table(frame):
    d = frame.describe().T
    out = d.rename(columns={"count": "N", "mean": "Mean", "std": "SD", "50%": "Median (50%)", "25%": "Q1 (25%)", "75%": "Q3 (75%)", "min": "Min", "max": "Max"})
    out["IQR (Q3-Q1)"] = out["Q3 (75%)"] - out["Q1 (25%)"]
    return out[["N", "Mean", "SD", "Median (50%)", "Q1 (25%)", "Q3 (75%)", "IQR (Q3-Q1)", "Min", "Max"]]

descriptive_table = summary_table(df[core_variables])
distribution_diagnostics = pd.DataFrame({
    "Mean": df[core_variables].mean(),
    "Median": df[core_variables].median(),
    "SD": df[core_variables].std(),
    "Skewness": df[core_variables].skew(),
    "Excess Kurtosis": df[core_variables].kurt(),
})
display(descriptive_table.round(3))
display(distribution_diagnostics.round(3))

In [ ]:
# Descriptive diagnostics only: ordinal overall-use counts and composite distributions.
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, col, title in zip(axes.flat,
        core_variables,
        ["GAD-7 total", "AI self-efficacy mean", "Overall LLM-use frequency", "Mean coding-task LLM use"]):
    if col == "llm_overall":
        counts = df[col].value_counts().reindex(range(1, 6), fill_value=0)
        ax.bar(counts.index, counts.values)
        ax.set_xticks(range(1, 6))
    else:
        ax.hist(df[col].to_numpy(), bins="auto")
    ax.set_title(title)
    ax.set_xlabel(col)
    ax.set_ylabel("Participants")
fig.tight_layout()
plt.show()

# Q-Q plots are diagnostics, not pass/fail normality tests.
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, col in zip(axes, ["gad7_total", "aise_mean", "llm_task_mean"]):
    stats.probplot(df[col], dist="norm", plot=ax)
    ax.set_title(f"Q-Q: {col}")
fig.tight_layout()
plt.show()

In [ ]:
llm_task_labels = {
    "llm_debug": "Debugging code",
    "llm_errors": "Understanding error messages",
    "llm_generate": "Generating code snippets",
    "llm_refactor": "Refactoring existing code",
    "llm_validate": "Editing/validating AI-generated code",
}
frequency_labels = {1: "Never", 2: "Rarely", 3: "Sometimes", 4: "Often", 5: "Always"}
llm_task_summary = summary_table(df[llm_task_variables])
llm_task_summary.insert(0, "Task", llm_task_summary.index.map(llm_task_labels))
# Percentages use all complete responses for each task as the denominator.
task_frequency_percentages = pd.DataFrame({
    col: df[col].value_counts(normalize=True).reindex(range(1, 6), fill_value=0).mul(100)
    for col in llm_task_variables
}).T.rename(columns=frequency_labels)
display(llm_task_summary.round(2))
display(task_frequency_percentages.round(1))

In [ ]:
# Preliminary association diagnostics. P-values and confirmatory tests are in Notebook 03.
pearson_correlations = df[core_variables].corr(method="pearson")
spearman_correlations = df[core_variables].corr(method="spearman")
relationship_pairs = [
    ("gad7_total", "llm_overall"),
    ("gad7_total", "llm_task_mean"),
    ("gad7_total", "aise_mean"),
    ("aise_mean", "llm_overall"),
    ("aise_mean", "llm_task_mean"),
]
correlation_comparison = pd.DataFrame([
    {
        "Variable 1": a,
        "Variable 2": b,
        "Pearson r": pearson_correlations.loc[a, b],
        "Spearman rho": spearman_correlations.loc[a, b],
        "Absolute difference": abs(pearson_correlations.loc[a, b] - spearman_correlations.loc[a, b]),
    }
    for a, b in relationship_pairs
])
display(pearson_correlations.round(3))
display(spearman_correlations.round(3))
display(correlation_comparison.round(3))

In [ ]:
# Visualize the principal RQ1 relationships without altering source values.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(df["gad7_total"], df["llm_task_mean"], alpha=0.5)
axes[0].set(xlabel="GAD-7 total", ylabel="Mean coding-task LLM use", title="Anxiety × task-specific use")
# Jitter is purely graphical and never enters statistical calculations.
rng = np.random.default_rng(42)
y_jitter = df["llm_overall"].to_numpy() + rng.normal(0, 0.05, len(df))
axes[1].scatter(df["gad7_total"], y_jitter, alpha=0.5)
axes[1].set(xlabel="GAD-7 total", ylabel="Overall LLM-use frequency", title="Anxiety × overall use", yticks=range(1, 6))
fig.tight_layout()
plt.show()

## Methodological interpretation

- `gad7_total`: summed GAD-7 score (0–21), treated as a scale score.
- `aise_mean`: mean of six seven-point self-efficacy items (1–7).
- `llm_overall`: a **single ordinal five-category** frequency item; Spearman is especially appropriate as a primary association check.
- `llm_task_mean`: mean of five frequency items, treated as an approximately continuous composite subject to diagnostic review.

The plots and correlation matrices describe the sample; they do **not** test RQ1 or RQ2. The pre-specified inferential choices and p-values belong in `03_quantitative_analysis.ipynb`. Outlying but valid observations are not automatically removed.

In [ ]:
# Final sanity check against the descriptives already verified in Notebook 01.
reference = {"gad7_total": (8.00, 5.85), "aise_mean": (4.99, 0.89),
             "llm_overall": (3.45, 1.22), "llm_task_mean": (3.15, 1.08)}
for variable, (expected_mean, expected_sd) in reference.items():
    actual_mean = descriptive_table.loc[variable, "Mean"]
    actual_sd = descriptive_table.loc[variable, "SD"]
    if round(actual_mean + 1e-10, 2) != expected_mean or round(actual_sd + 1e-10, 2) != expected_sd:
        raise AssertionError(f"Mismatch for {variable}: mean={actual_mean:.5f}, SD={actual_sd:.5f}")
print("PASS: all four descriptive mean/SD pairs match the thesis to 2 decimals.")
print("Exploratory analysis completed. No participant-level data exported.")